# Sesión 2: Análisis y matemática del LTEE
Datos: Wiser, Ribeck y Lenski (2013), *Science* 342: 1364-1367. Repositorio Dryad, doi:10.5061/dryad.0hc2m.
Archivo: `Concatenated.LTEE.data.all.csv`.

In [ ]:
# Celda 1: librerías y carga del archivo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from google.colab import files
files.upload()   # seleccionar Concatenated.LTEE.data.all.csv

datos_brutos = pd.read_csv("Concatenated.LTEE.data.all.csv")
print(datos_brutos.shape)        # (928, 14): 928 registros y 14 columnas
datos_brutos.head()

In [ ]:
# Celda 2: revisar los datos antes de limpiarlos
print(datos_brutos.columns.tolist())
print("Poblaciones:", datos_brutos["Population"].nunique())
print("Generaciones distintas:", datos_brutos["Generation"].nunique())
print("Valores vacíos de Fitness:", datos_brutos["Fitness"].isna().sum())
datos_brutos[datos_brutos["Fitness"].isna()]

In [ ]:
# Celda 3: limpieza (los datos brutos no se modifican)
datos = datos_brutos.copy()
datos = datos[["Generation", "Population", "Rep", "Fitness", "Mutator.Ever"]]
datos = datos.rename(columns={"Generation": "t", "Population": "poblacion",
                              "Rep": "replica", "Fitness": "w",
                              "Mutator.Ever": "hipermutadora"})
datos = datos.dropna(subset=["w"])            # se eliminan los 6 ensayos sin aptitud
print("Registros después de limpiar:", len(datos))   # 922

In [ ]:
# Celda 4: promedio por población y promedio general
# Paso 1: promediar las réplicas de cada población en cada generación
por_poblacion = datos.groupby(["t", "poblacion", "hipermutadora"], as_index=False)["w"].mean()

# Paso 2: promediar las poblaciones en cada generación (promedio general)
general = por_poblacion.groupby("t").agg(w=("w", "mean"), n_poblaciones=("w", "count")).reset_index()
general

In [ ]:
# Celda 5: tabla de tasas de cambio promedio (pendiente de la secante)
cortes = [0, 10000, 20000, 30000, 40000, 50000]
w_en = general.set_index("t")["w"]
n_en = general.set_index("t")["n_poblaciones"]
filas = []
for t1, t2 in zip(cortes[:-1], cortes[1:]):
    tasa = (w_en[t2] - w_en[t1]) / (t2 - t1)
    filas.append([f"{t1} a {t2}", round(w_en[t1], 4), round(w_en[t2], 4), f"{tasa:.2e}",
                  f"{n_en[t1]} y {n_en[t2]}"])
tabla_tasas = pd.DataFrame(filas, columns=["Intervalo", "w(t1)", "w(t2)",
                                           "Tasa promedio (gen^-1)", "Poblaciones promediadas"])
tabla_tasas

In [ ]:
# Celda 6: Gráfica 1, comparación de dos grupos
grupos = por_poblacion.groupby(["t", "hipermutadora"]).agg(w=("w", "mean"), n=("w", "count")).reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
for valor, etiqueta, color, marcador in [("No", "Tasa de mutación ancestral (6 poblaciones)", "#2b6cb0", "o"),
                                         ("Yes", "Hipermutadoras en algún momento (6, luego 3)", "#dd6b20", "^")]:
    g = grupos[grupos["hipermutadora"] == valor]
    ax.plot(g["t"], g["w"], marker=marcador, ms=5, lw=1.5, color=color, label=etiqueta)
ax.axhline(1, color="gray", lw=0.8, ls=":")
ax.set_title("Gráfica 1. Aptitud promedio por grupo de poblaciones (LTEE)")
ax.set_xlabel("Tiempo t (generaciones)")
ax.set_ylabel("Aptitud relativa w (sin unidades)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
fig.text(0.01, 0.01, "Fuente: Wiser et al. (2013), Dryad doi:10.5061/dryad.0hc2m", fontsize=7, color="gray")
plt.tight_layout()
plt.savefig("grafica1_grupos.png", dpi=110)
plt.show()

In [ ]:
# Celda 7: ajuste de tres modelos y comparación con R²
t = general["t"].to_numpy(dtype=float)
w = general["w"].to_numpy()

def lineal(t, m, c):       return m * t + c
def exponencial(t, A, k):  return 1 + A * (1 - np.exp(-k * t))   # exponencial acotada
def potencia(t, a, b):     return (b * t + 1) ** a

modelos = {"Lineal": (lineal, [1e-5, 1.3]),
           "Exponencial acotada": (exponencial, [0.7, 2e-4]),
           "Ley de potencia": (potencia, [0.1, 0.005])}
parametros, r2 = {}, {}
for nombre, (f, inicio) in modelos.items():
    p, _ = curve_fit(f, t, w, p0=inicio, maxfev=20000)
    parametros[nombre] = p
    residuos = w - f(t, *p)
    r2[nombre] = 1 - np.sum(residuos**2) / np.sum((w - w.mean())**2)
    print(f"{nombre:20s} parámetros = {p}   R² = {r2[nombre]:.3f}")

In [ ]:
# Celda 8: Gráfica 2, los tres modelos sobre los datos
tt = np.linspace(0, 50000, 500)
estilos = {"Lineal": ("#c53030", "--"), "Exponencial acotada": ("#805ad5", "-."),
           "Ley de potencia": ("#2b6cb0", "-")}
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(t, w, color="black", s=18, zorder=3, label="Promedio general (datos)")
for nombre, (f, _) in modelos.items():
    color, linea = estilos[nombre]
    ax.plot(tt, f(tt, *parametros[nombre]), color=color, ls=linea, lw=2,
            label=f"{nombre} (R² = {r2[nombre]:.2f})")
ax.set_title("Gráfica 2. Ajuste de modelos a la aptitud promedio")
ax.set_xlabel("Tiempo t (generaciones)")
ax.set_ylabel("Aptitud relativa w (sin unidades)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
fig.text(0.01, 0.01, "Fuente: Wiser et al. (2013), Dryad doi:10.5061/dryad.0hc2m", fontsize=7, color="gray")
plt.tight_layout()
plt.savefig("grafica2_modelos.png", dpi=110)
plt.show()

In [ ]:
# Celda 9: derivada del modelo de potencia y secantes que se acercan a P
a, b = parametros["Ley de potencia"]
a, b = round(a, 4), round(b, 5)        # valores redondeados que usamos en el tablero
W = lambda t: (b * t + 1) ** a
dW = lambda t: a * b * (b * t + 1) ** (a - 1)    # regla de la cadena

t0 = 10000
print("Punto P:", t0, round(W(t0), 4))
print("Derivada en t = 10000:", f"{dW(t0):.3e}")
print("Derivada en t = 0:", f"{dW(0):.3e}")
secantes = []
for tq in [20000, 15000, 11000, 10100, 10010]:
    m = (W(tq) - W(t0)) / (tq - t0)
    secantes.append([tq, round(W(tq), 4), f"{m:.3e}"])
pd.DataFrame(secantes, columns=["t de Q", "w(Q)", "pendiente PQ (gen^-1)"])

In [ ]:
# Celda 10: Gráfica 3, recta secante y recta tangente
t1, t2 = 0, 20000                                   # puntos de la secante
m_sec = (W(t2) - W(t1)) / (t2 - t1)                 # pendiente de la secante (tasa promedio)
m_tan = dW(t0)                                      # pendiente de la tangente (derivada en t0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(tt, W(tt), color="#2b6cb0", lw=3, label="Modelo de aptitud w(t)")
xs = np.array([0, 25000])
ax.plot(xs, W(t1) + m_sec * (xs - t1), color="#e53e3e", lw=2, ls="--",
        label=f"Recta secante (0 a 20.000 gen) [tasa media: {m_sec:.2e}]")
ax.plot([t1, t2], [W(t1), W(t2)], "o", color="#e53e3e", ms=9, zorder=4)
xs = np.array([2000, 18000])
ax.plot(xs, W(t0) + m_tan * (xs - t0), color="#38a169", lw=2.5,
        label=f"Recta tangente en t = 10.000 gen [tasa instantánea (derivada): {m_tan:.2e}]")
ax.plot(t0, W(t0), "o", color="#38a169", ms=9, zorder=5)
ax.set_title("Gráfica 3. Representación geométrica de secante, tangente y derivada")
ax.set_xlabel("Tiempo t (generaciones)")
ax.set_ylabel("Aptitud relativa w (sin unidades)")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
fig.text(0.01, 0.01, "Modelo ajustado a datos de Wiser et al. (2013)", fontsize=7, color="gray")
plt.tight_layout()
plt.savefig("grafica3_secante_tangente.png", dpi=110)
plt.show()